In [1]:
import yfinance as yf
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# =====================
# BAIXAR DADOS
# =====================

ibov = yf.download("^BVSP", start="2010-01-01", progress=False)["Close"]
usd_brl = yf.download("USDBRL=X", start="2010-01-01", progress=False)["Close"]

# =====================
# PREPARAR SÉRIES
# =====================

df = pd.concat([ibov, usd_brl], axis=1)
df.columns = ["Ibov", "USD_BRL"]
df = df.dropna()
df["Ibov_USD"] = df["Ibov"] / df["USD_BRL"]
df_norm = df[["Ibov", "Ibov_USD", "USD_BRL"]] / df[["Ibov", "Ibov_USD", "USD_BRL"]].iloc[0] * 100

# =====================
# DISTÂNCIA DA REGRESSÃO LINEAR (%)
# =====================

x = np.arange(len(df_norm))
ativos = ["Ibov_USD", "USD_BRL"]

for ativo in ativos:
    y = df_norm[ativo].values
    coef = np.polyfit(x, y, 1)
    reg = coef[0] * x + coef[1]
    df_norm[f"{ativo}_desvio"] = ((y - reg) / reg) * 100

# =====================
# PLOT: LINHA DE DESVIO + HISTOGRAMA
# =====================

cores = {
    "Ibov_USD": "#2ca02c",
    "USD_BRL": "#d62728",
}

fig = make_subplots(
    rows=1,
    cols=2,
    column_widths=[0.72, 0.28],
    horizontal_spacing=0.04,
)

for ativo in ativos:
    serie_desvio = df_norm[f"{ativo}_desvio"]
    valor_atual = serie_desvio.dropna().iloc[-1]

    fig.add_trace(
        go.Scatter(
            x=df_norm.index,
            y=serie_desvio,
            mode="lines",
            name=ativo,
            line=dict(width=2, color=cores[ativo]),
            hovertemplate="%{x|%d/%m/%Y}<br>%{y:.2f}%<extra>" + ativo + "</extra>"
        ),
        row=1,
        col=1,
    )

    fig.add_trace(
        go.Histogram(
            x=serie_desvio.dropna(),
            name=ativo,
            marker_color=cores[ativo],
            opacity=0.45,
            nbinsx=70,
            showlegend=False,
            hovertemplate="Desvio: %{x:.2f}%<br>Frequência: %{y}<extra>" + ativo + "</extra>"
        ),
        row=1,
        col=2,
    )

    fig.add_vline(
        x=valor_atual,
        line_dash="dot",
        line_width=1,
        line_color=cores[ativo],
        opacity=0.45,
        row=1,
        col=2,
    )

    fig.add_trace(
        go.Scatter(
            x=[valor_atual],
            y=[0],
            mode="markers",
            marker=dict(size=8, color=cores[ativo], symbol="diamond-open", line=dict(width=1)),
            showlegend=False,
            hovertemplate="Atual: %{x:.2f}%<extra>" + ativo + "</extra>"
        ),
        row=1,
        col=2,
    )

fig.add_hline(y=0, line_dash="dash", line_color="gray", row=1, col=1)
fig.add_vline(x=0, line_dash="dash", line_color="gray", row=1, col=2)

fig.update_layout(
    title="Desvio da Regressão Linear (%) - Ibov em USD e USD/BRL",
    template="plotly_white",
    height=650,
    barmode="overlay",
    legend_title=None,
    title_x=0.5
)

fig.update_xaxes(title_text="Data", showgrid=True, row=1, col=1)
fig.update_yaxes(title_text="Desvio da Regressão (%)", showgrid=True, row=1, col=1)
fig.update_xaxes(title_text="Desvio (%)", showgrid=True, row=1, col=2)
fig.update_yaxes(title_text="Frequência", showgrid=True, row=1, col=2)

fig.show()
